In [1]:
import os
import sys
sys.path.append("../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data, Embedding
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune_head import (
    compute_heads_importance,
    head_importance_prunning,
)
from utils.prune_utils.prune import recover_tangling_identification, prune_concern_identification, prune_wanda

In [3]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from torch.nn import CrossEntropyLoss, MSELoss
from functools import partial
import torch.nn.functional as F
import math
from sklearn.metrics import classification_report
import gc

In [4]:
name= "YahooAnswersTopics"
device = torch.device("cuda:0")
checkpoint = None
batch_size=16
num_workers=4
seed=44

In [5]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'fabriceyhc/bert-base-uncased-yahoo_answers_topics', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}
The model fabriceyhc/bert-base-uncased-yahoo_answers_topics is loaded.


In [6]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    name, batch_size=batch_size, num_workers=num_workers, do_cache=True, seed=seed
)

Loading cached dataset YahooAnswersTopics.
The dataset YahooAnswersTopics is loaded
{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/Yahoo', 'task_type': 'classification'}


In [7]:
def extract_top_k_embeddings(extract_num, class_num, embeddings, attention_mask):
    extracted_embeddings_tensor = torch.stack([embeddings[i][:extract_num] for i in range(class_num)])
    extracted_attention_mask_tensor = torch.stack([attention_mask[i][:extract_num] for i in range(class_num)])
    
    return extracted_embeddings_tensor, extracted_attention_mask_tensor

In [8]:
def get_decimal_precision(value):
    str_value = str(value)
    if '.' in str_value:
        return len(str_value.split('.')[1])
    else:
        return 0

In [9]:
def perturb(embeddings, eps, grad):
    perturbed_embeddings = embeddings - eps*grad.sign()
    return perturbed_embeddings

In [10]:
def calculate_gradient(model, input_embeddings, attention_mask, target_class, device):
    input_embeddings = input_embeddings.to(device)
    attention_mask = attention_mask.to(device)

    batch_size = input_embeddings.size(0)
    losses = []
    gradients = []

    for i in range(batch_size):
        embedding = input_embeddings[i:i+1].requires_grad_(True)
        mask = attention_mask[i:i+1]
        target = torch.tensor([target_class], device=device)
        
        outputs = model(inputs_embeds=embedding, attention_mask=mask, labels=target)
        loss = outputs.loss
        
        model.zero_grad()
        loss.backward()
        losses.append(loss.item())
        gradients.append(embedding.grad.squeeze(0).detach())

    
    stacked_gradients = torch.stack(gradients)
    return losses, stacked_gradients

In [11]:
def fgsm_attack(model, target_class, input_embeddings, attention_mask, start_epsilon, epsilon_step, max_epsilon, device):
    input_embeddings = input_embeddings.to(device)
    attention_mask = attention_mask.to(device)
    digits = get_decimal_precision(epsilon_step)
    eps = start_epsilon
    loss, data_grad = calculate_gradient(model, input_embeddings, attention_mask,target_class, device)
    
    while eps <= max_epsilon:
        perturbed_embeddings = perturb(input_embeddings, eps, data_grad)
        with torch.no_grad():
            adv_outputs = model(inputs_embeds=perturbed_embeddings, attention_mask=attention_mask)
            adv_pred = adv_outputs.logits.argmax(dim=-1)
        
        if adv_pred.item() == target_class:
            break
        else:
            eps += epsilon_step
            eps = round(eps, digits)
    return eps, perturbed_embeddings

In [12]:
def calculate_all_epsilon(model, class_num, top_k_embeddings, attention_mask, step_epsilon = 0.01, max_epsilon = 10.0):
    epsilon_list = []
    max_eps = max_epsilon
    batch_size = top_k_embeddings[0].shape[0]
    for i in range(class_num):
        temp = []
        
        for j in range(class_num):
            if i == j:
                temp.append([float("inf")] * batch_size)
                continue
            epsilon_per_batch = []
            for b in range(batch_size):
                embedding = top_k_embeddings[i][b:b+1]
                mask = attention_mask[i][b:b+1]
                epsilon, perterbed = fgsm_attack(
                    model, j, embedding, mask, 0.00, step_epsilon, max_eps, device
                )
                if epsilon >= max_eps:
                    epsilon_per_batch.append(float("inf"))
                else:
                    epsilon_per_batch.append(epsilon)
            temp.append(epsilon_per_batch)
        epsilon_list.append(temp)
    return epsilon_list

In [13]:
def select_true_example(model, num_labels, data_loader, tokenizer, device):
    correct_predictions = [[] for _ in range(num_labels)]
    
    for batch in tqdm(data_loader, desc="Collecting correct predictions"):
        inputs = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        with torch.no_grad():
            outputs = model(inputs, attention_mask=attention_mask, output_hidden_states=True)
            input_embeddings, output_embeddings = outputs.hidden_states[0], outputs.hidden_states[-1]

        predictions = outputs.logits.argmax(dim=-1)
        correct_indices = (predictions == labels).nonzero(as_tuple=True)[0]
        
        for i in correct_indices:
            label = labels[i].item()
            logits = outputs.logits[i, predictions[i]].item()
            
            correct_predictions[label].append((
                logits,
                inputs[i].cpu(),
                input_embeddings[i].cpu(),
                output_embeddings[i].cpu(),
                attention_mask[i].cpu()
            ))
        del inputs, labels, attention_mask, outputs, input_embeddings, output_embeddings, predictions, correct_indices
        torch.cuda.empty_cache()
                
    sorted_inputs = []
    sorted_input_embeddings = []
    sorted_output_embeddings = []
    sorted_attention_mask = []
    
    for predictions in correct_predictions:
        predictions.sort(key=lambda x: x[0], reverse=True)
        
        sorted_inputs.append(torch.stack([pred[1] for pred in predictions]))
        sorted_input_embeddings.append(torch.stack([pred[2] for pred in predictions]))
        sorted_output_embeddings.append(torch.stack([pred[3] for pred in predictions]))
        sorted_attention_mask.append(torch.stack([pred[4] for pred in predictions]))
    
    return sorted_inputs, sorted_input_embeddings, sorted_output_embeddings, sorted_attention_mask

In [14]:
def generate_example(model, start_epsilon, end_epsilon, source_class, target_class, input_embeddings, attention_mask, device, per_attack_example_num, min_step_eps=1e-5):
    example_list = []
    example_label = []
    attention_mask_list = []
    
    source_embedding = input_embeddings[source_class].to(device)
    source_attention_mask = attention_mask[source_class].to(device)
    loss, data_grad = calculate_gradient(model, source_embedding, source_attention_mask, target_class, device)

    step_eps = (end_epsilon - start_epsilon) / per_attack_example_num
    step_eps = max(step_eps, min_step_eps)
    iter_eps = start_epsilon
    iter_num = 0
    
    while iter_eps < end_epsilon and iter_num < per_attack_example_num:
        generated_embeddings = perturb(source_embedding, iter_eps, data_grad)
        batch_size = generated_embeddings.shape[0]
        for b in range(batch_size):
            example_list.append(generated_embeddings[b].cpu())
            attention_mask_list.append(source_attention_mask[b].cpu())
        example_label.extend([source_class] * batch_size)
        
        iter_eps += step_eps
        iter_num += 1
        
        del generated_embeddings
        torch.cuda.empty_cache()
        
    del source_embedding, source_attention_mask, loss, data_grad
    torch.cuda.empty_cache()
    
    return example_label, example_list, attention_mask_list

In [15]:
def make_example(model, data_loader, tokenizer, example_num, emb_num, class_num, true_ratio, device, step_epsilon=0.01, max_epsilon=10.0):
    positive_example_list = []
    positive_example_label = []
    positive_attention_mask = []
    
    negative_example_list = []
    negative_example_label = []
    negative_attention_mask = []
    
    positive_num = int(example_num * true_ratio)
    negative_num = example_num - positive_num
    
    print("positive num : ", positive_num)
    print("negative num : ", negative_num)
    
    inputs, input_embeddings, output_embeddings, attention_mask = select_true_example(model, class_num, data_loader, tokenizer, device)
    extracted_embeddings, extracted_attention_mask = extract_top_k_embeddings(emb_num, class_num, input_embeddings, attention_mask)
    
    epsilon_list = calculate_all_epsilon(model, class_num, extracted_embeddings, extracted_attention_mask, step_epsilon, max_epsilon)
    
    per_class_positive_example_num = int(positive_num / class_num)
    per_class_negative_example_num = int(negative_num / class_num)
    print("per_class_positive_example_num : ", per_class_positive_example_num)
    print("per_class_negative_example_num : ", per_class_negative_example_num)
    
    for source_class in range(class_num):
        inf_num = sum(all(item == float('inf') for item in epsilon) for epsilon in epsilon_list[source_class])
        per_target_positive_example_num = int(per_class_positive_example_num / (class_num - inf_num))
        per_target_negative_example_num = int(per_class_negative_example_num / (class_num - inf_num))
        
        for target_class in range(class_num):
            epsilon = epsilon_list[source_class][target_class]

            if all(eps == float("inf") for eps in epsilon):
                continue

            for eps in epsilon:
                if eps == float("inf"):
                    continue
                    
                pos_label, pos_examples, pos_mask = generate_example(
                    model,
                    start_epsilon=0.00,
                    end_epsilon=eps - step_epsilon,
                    source_class=source_class,
                    target_class=target_class,
                    input_embeddings=extracted_embeddings,
                    attention_mask=extracted_attention_mask,
                    device=device,
                    per_attack_example_num=per_target_positive_example_num
                )
                positive_example_label.extend(pos_label)
                positive_example_list.extend(pos_examples)
                positive_attention_mask.extend(pos_mask)
                
                # Negative 예제 생성
                neg_label, neg_examples, neg_mask = generate_example(
                    model,
                    start_epsilon=eps,
                    end_epsilon=2 * eps - step_epsilon,
                    source_class=source_class,
                    target_class=target_class,
                    input_embeddings=extracted_embeddings,
                    attention_mask=extracted_attention_mask,
                    device=device,
                    per_attack_example_num=per_target_negative_example_num
                )
                negative_example_label.extend(neg_label)
                negative_example_list.extend(neg_examples)
                negative_attention_mask.extend(neg_mask)

        del pos_label, pos_examples, pos_mask, neg_label, neg_examples, neg_mask
        torch.cuda.empty_cache()
        gc.collect()
        
    pos_embeddings = Embedding(positive_example_list, positive_example_label, positive_attention_mask)
    neg_embeddings = Embedding(negative_example_list, negative_example_label, negative_attention_mask)

    del positive_example_list, positive_example_label, positive_attention_mask
    del negative_example_list, negative_example_label, negative_attention_mask
    torch.cuda.empty_cache()
    gc.collect()
    
    return pos_embeddings, neg_embeddings

In [16]:
positive_embeddings, negative_embeddings = make_example(
    model, data_loader=valid_dataloader,
    tokenizer=tokenizer, example_num=3000,
    emb_num=1, class_num=10, true_ratio=0.5,
    device=device, step_epsilon=0.01, max_epsilon=10.0
)

positive num :  1500
negative num :  1500


per_class_positive_example_num :  150
per_class_negative_example_num :  150


In [17]:
pos_dataloader = DataLoader(positive_embeddings, batch_size=4, shuffle=False, num_workers=16)
neg_dataloader = DataLoader(negative_embeddings, batch_size=4, shuffle=False, num_workers=16)

In [18]:
batch_size = 16
num_workers = 4
num_samples = 128
ci_ratio = 0.6
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [19]:
pos_dataloader = SamplingDataset(
    pos_dataloader, 0, num_samples, num_labels, True, 4, device=device, resample=False, seed=seed
)
neg_dataloader = SamplingDataset(
    neg_dataloader, 200, num_samples, num_labels, False, 4, device=device, resample=False, seed=seed
)

In [20]:
# sampling 128 examples, in the concern 0

In [21]:
module = copy.deepcopy(model)

In [22]:
prune_concern_identification(
    module,
    model_config,
    pos_dataloader,
    neg_dataloader,
    include_layers=include_layers,
    exclude_layers=exclude_layers,
    sparsity_ratio=ci_ratio,
)

In [23]:
result = evaluate_model(module, model_config, test_dataloader)

Evaluating the model:   0%|                            | 0/1875 [00:00<?, ?it/s]

Loss: 1.5574
Precision: 0.6445, Recall: 0.5450, F1-Score: 0.5489
              precision    recall  f1-score   support

           0       0.51      0.52      0.52      2941
           1       0.85      0.24      0.37      2997
           2       0.84      0.44      0.57      3016
           3       0.53      0.37      0.44      2978
           4       0.76      0.74      0.75      3017
           5       0.97      0.58      0.72      3004
           6       0.29      0.46      0.36      3037
           7       0.38      0.76      0.51      3026
           8       0.49      0.82      0.61      2997
           9       0.83      0.52      0.64      2987

    accuracy                           0.55     30000
   macro avg       0.64      0.54      0.55     30000
weighted avg       0.64      0.55      0.55     30000



In [24]:
# Loss: 1.5576
# Precision: 0.6446, Recall: 0.5452, F1-Score: 0.5489
#               precision    recall  f1-score   support

#            0       0.51      0.52      0.51      2941
#            1       0.84      0.23      0.37      2997
#            2       0.84      0.44      0.57      3016
#            3       0.54      0.37      0.44      2978
#            4       0.76      0.75      0.75      3017
#            5       0.97      0.58      0.72      3004
#            6       0.29      0.46      0.36      3037
#            7       0.38      0.76      0.51      3026
#            8       0.49      0.82      0.61      2997
#            9       0.83      0.52      0.64      2987

#     accuracy                           0.55     30000
#    macro avg       0.64      0.55      0.55     30000
# weighted avg       0.64      0.55      0.55     30000